# nb127 — Chemprop multi-target pretrain on Papyrus, fine-tune on PXR (Kaggle GPU)

Use the 19,948 Papyrus++ records covering 41 PXR-related targets (NRs, P450s, transporters) to pretrain a Chemprop MPNN with 41 output heads. Then fine-tune on the 4,139 PXR CRC compounds.

Goal: produce a NEW base model whose OOF/test predictions can be added to the nb224 SLSQP pool. The transfer-learning signal from related targets should give predictions that are diverse from existing LGBM/XGB anchors.

Architecture: BondMessagePassing depth=4, d_h=300, MeanAgg, FFN(2 layers, 0.1 dropout). Multi-task: mask NaN targets per compound.

Outputs:
  - oof_nb127_chemprop_papyrus.npy (scaffold 5-fold OOF on PXR train)
  - te_nb127_chemprop_papyrus.npy (test predictions)
  - Optional: encoder embeddings (300-dim per compound)

In [ ]:
import subprocess, sys, os
from pathlib import Path
os.environ["PYTHONUNBUFFERED"] = "1"
# Force CPU — P100 (sm_60) on Kaggle isn't compatible with bundled PyTorch.
# Reimporting torch after pip install corrupts module state.
ACCEL = "cpu"
print(f"Using accelerator: {ACCEL} (P100 incompatibility avoided)")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chemprop>=2.0.0", "rdkit", "lightning"], check=False)
import torch
print(f"torch: {torch.__version__}  cuda_available={torch.cuda.is_available()}")
print("installs done")

In [ ]:
import pandas as pd, numpy as np, glob, os, urllib.request

# Try several possible Kaggle dataset mount paths
papy_path = None
search_patterns = [
    "/kaggle/input/**/papyrus_wide_compound_x_target.parquet",
    "/kaggle/input/**/papyrus_pxr_related_filtered.parquet",
]
for pattern in search_patterns:
    matches = glob.glob(pattern, recursive=True)
    if matches:
        for m in matches:
            print(f"found: {m}")
        papy_path = matches[0]
        break

if not papy_path:
    print("Papyrus parquet not in Kaggle dataset; running quick filter locally...")
    import subprocess, sys
    subprocess.run([sys.executable,"-m","pip","install","-q","papyrus-scripts"], check=False)
    from papyrus_scripts import download_papyrus
    from papyrus_scripts.reader import read_papyrus
    from papyrus_scripts.preprocess import keep_accession
    print("Downloading Papyrus++ 05.7...")
    download_papyrus(version="05.7", only_pp=True, structures=False, descriptors=None)
    targets = ["O75469","Q14994","P11473","Q96RI1","Q13133","P55055","Q07869","P37231","Q03181",
               "P10276","P10826","P13631","P19793","P28702","P48443","P10275","P03372","Q92731",
               "P04150","P08235","P10827","P10828","P41235","P11474","O95718","P62508",
               "P08684","P11712","P33261","P05177","P10635","P05181","P20815","P20813",
               "P08183","Q9UNQ0","Q92887","Q9Y6L6","Q9NPD5","P35869","P02768"]
    filtered = []
    for i, chunk in enumerate(read_papyrus(version="05.7", plusplus=True, is3d=False, chunksize=200_000)):
        sub = keep_accession(chunk, targets)
        if len(sub) > 0:
            filtered.append(sub)
        if i % 10 == 0: print(f"  chunk {i}: {sum(len(c) for c in filtered):,} so far")
    papy = pd.concat(filtered, ignore_index=True)
    value_col = "pchembl_value_Mean"
    smi_col = "SMILES" if "SMILES" in papy.columns else "SMILES_Stripped"
    wide = papy.pivot_table(index=smi_col, columns="accession", values=value_col, aggfunc="median").reset_index()
    print(f"Built Papyrus wide locally: {wide.shape}")
else:
    wide = pd.read_parquet(papy_path)
    print(f"Wide papyrus loaded: {wide.shape}")

target_cols = [c for c in wide.columns if c not in ("SMILES","SMILES_Stripped","connectivity","index")]
smi_col = "SMILES" if "SMILES" in wide.columns else ("SMILES_Stripped" if "SMILES_Stripped" in wide.columns else "connectivity")
print(f"SMILES col: {smi_col}, target cols ({len(target_cols)}): {target_cols[:10]}")
wide = wide.dropna(subset=[smi_col])
print(f"After dropna SMILES: {len(wide)} unique compounds")


In [ ]:
# Load PXR train + test
HF = 'https://huggingface.co/datasets/openadmet/pxr-challenge-train-test/resolve/main'
TR_LOC = '/kaggle/working/train.csv'; TE_LOC = '/kaggle/working/test.csv'
if not Path(TR_LOC).exists():
    urllib.request.urlretrieve(f'{HF}/pxr-challenge_TRAIN.csv', TR_LOC)
    urllib.request.urlretrieve(f'{HF}/pxr-challenge_TEST_BLINDED.csv', TE_LOC)
tr = pd.read_csv(TR_LOC); te = pd.read_csv(TE_LOC)
print(f'Train: {len(tr)}  Test: {len(te)}')

In [ ]:
# Build Chemprop multi-target dataset: each row = (smiles, [t1, t2, ..., t41])
# Use chemprop 2.x API
from chemprop import data, featurizers, models
from chemprop import nn as cnn
from lightning import pytorch as L
import torch

# Reshape wide into Chemprop format
smiles_papy = wide[smi_col].tolist()
y_papy = wide[target_cols].to_numpy(dtype=np.float32)  # NaN where target not measured
print(f'Papyrus: {len(smiles_papy)} compounds x {y_papy.shape[1]} targets')
print(f'  NaN fraction: {np.isnan(y_papy).mean()*100:.1f}%')

In [ ]:
# Build pretraining datapoints (multi-task with NaN masking)
datapoints_pretrain = []
for s, y in zip(smiles_papy, y_papy):
    try:
        datapoints_pretrain.append(data.MoleculeDatapoint.from_smi(s, y))
    except Exception:
        pass
print(f'Built {len(datapoints_pretrain)} pretrain datapoints')

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
ds_pretrain = data.MoleculeDataset(datapoints_pretrain, featurizer)
loader_pretrain = data.build_dataloader(ds_pretrain, batch_size=64, num_workers=0, shuffle=True)
print(f'Dataset ready')

In [ ]:
# Build a multi-target Chemprop model — reduced size for CPU tractability
n_tasks = y_papy.shape[1]
mp = cnn.BondMessagePassing(depth=3, d_h=200, dropout=0.1)
agg = cnn.MeanAggregation()
ffn = cnn.RegressionFFN(n_tasks=n_tasks, input_dim=mp.output_dim, hidden_dim=200, n_layers=2, dropout=0.1)
model = models.MPNN(mp, agg, ffn, batch_norm=True, metrics=[cnn.metrics.MAE()])

trainer = L.Trainer(
    max_epochs=12, accelerator=ACCEL,
    devices=1, gradient_clip_val=1.0, enable_progress_bar=False,
    logger=False, enable_checkpointing=False,
)

import time
t0 = time.time()
trainer.fit(model=model, train_dataloaders=loader_pretrain)
print(f'Pretrain done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Now fine-tune on PXR. Replace FFN with single-task regression head (warm-start MP encoder).
import torch.nn as tnn

# Build PXR train datapoints
pxr_dps_tr = []
for s, y in zip(tr['SMILES'], tr['pEC50']):
    if pd.isna(y) or pd.isna(s): continue
    try:
        pxr_dps_tr.append(data.MoleculeDatapoint.from_smi(s, np.array([y], dtype=np.float32)))
    except Exception:
        pass
pxr_dps_te = []
for s, n in zip(te['SMILES'], te['Molecule Name']):
    if pd.isna(s): continue
    try:
        pxr_dps_te.append(data.MoleculeDatapoint.from_smi(s, np.array([0.0], dtype=np.float32)))
    except Exception:
        pxr_dps_te.append(None)
print(f'PXR datapoints: train={len(pxr_dps_tr)} test={sum(d is not None for d in pxr_dps_te)}')

In [ ]:
# Scaffold 5-fold CV on PXR with the pretrained encoder
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import Chem

def murcko(smi):
    try:
        m = Chem.MolFromSmiles(smi)
        if m is None: return str(smi)
        return MurckoScaffold.MurckoScaffoldSmiles(mol=m)
    except Exception:
        return str(smi)

scaffolds = [murcko(s) for s in tr['SMILES']]
from collections import defaultdict
scaf2idx = defaultdict(list)
for i, s in enumerate(scaffolds): scaf2idx[s].append(i)
groups = sorted(scaf2idx.items(), key=lambda x: -len(x[1]))
fold_assign = np.zeros(len(scaffolds), dtype=int)
for fi, (scaf, idx) in enumerate(groups):
    f = fi % 5
    for i in idx: fold_assign[i] = f
print('fold sizes:', [(fold_assign==i).sum() for i in range(5)])

oof = np.zeros(len(pxr_dps_tr), dtype=np.float32)
te_preds = []

for fold in range(5):
    print(f'\n--- Fold {fold+1}/5 ---')
    val_mask = (fold_assign == fold)
    tr_mask = ~val_mask
    dp_tr = [pxr_dps_tr[i] for i in range(len(pxr_dps_tr)) if tr_mask[i]]
    dp_va = [pxr_dps_tr[i] for i in range(len(pxr_dps_tr)) if val_mask[i]]
    ds_tr = data.MoleculeDataset(dp_tr, featurizer)
    ds_va = data.MoleculeDataset(dp_va, featurizer)
    ds_te = data.MoleculeDataset([d for d in pxr_dps_te if d is not None], featurizer)
    ldr_tr = data.build_dataloader(ds_tr, batch_size=128, num_workers=0, shuffle=True)
    ldr_va = data.build_dataloader(ds_va, batch_size=128, num_workers=0, shuffle=False)
    ldr_te = data.build_dataloader(ds_te, batch_size=128, num_workers=0, shuffle=False)

    # PXR head warm-starts from pretrained MP encoder
    ffn_pxr = cnn.RegressionFFN(n_tasks=1, input_dim=mp.output_dim, hidden_dim=200, n_layers=2, dropout=0.1)
    model_pxr = models.MPNN(mp, agg, ffn_pxr, batch_norm=True, metrics=[cnn.metrics.MAE()])

    trainer_ft = L.Trainer(
        max_epochs=8, accelerator=ACCEL,
        devices=1, gradient_clip_val=1.0, enable_progress_bar=False,
        logger=False, enable_checkpointing=False,
    )
    trainer_ft.fit(model=model_pxr, train_dataloaders=ldr_tr, val_dataloaders=ldr_va)

    pred_va = torch.cat([model_pxr(b).cpu() for b in ldr_va]).numpy().flatten()
    pred_te = torch.cat([model_pxr(b).cpu() for b in ldr_te]).numpy().flatten()
    oof[val_mask] = pred_va[:val_mask.sum()]
    te_preds.append(pred_te)

te_pred = np.mean(te_preds, axis=0)
rae = np.abs(tr['pEC50'].values - oof).sum() / np.abs(tr['pEC50'].values - tr['pEC50'].mean()).sum()
ratio = te_pred.std() / oof.std()
print(f'\nFinal: OOF RAE={rae:.4f}  ratio={ratio:.3f}  te_std={te_pred.std():.3f}')

In [ ]:
# Save outputs
out_dir = Path('/kaggle/working')
np.save(out_dir / 'oof_nb127_chemprop_papyrus.npy', oof)
np.save(out_dir / 'te_nb127_chemprop_papyrus.npy', te_pred)
print(f'Saved oof/te for nb127  (OOF RAE={rae:.4f})')